# Caso 7 — MDS de 6 nodos: QAOA–COBYLA frente a fuerza bruta, ILP y Greedy

Experimento **real** (sin datos escritos a mano): todo sale de `qml_v2.mds_qaoa`, que guarda sus resultados en `data/output/metrics/10_mds_qaoa_*`. El simulador NumPy de QAOA se verifica contra `QAOAAnsatz` + `Statevector` de Qiskit.

**Nota**: la penalización cuadrática del QUBO codifica un *dominante perfecto*; coincide con el MDS usual solo si existe un dominante perfecto mínimo (cierto en esta instancia).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
from qml_v2 import mds_qaoa as m
reinicios, trayectorias, dist, info = m.run(verbose=False)
print('Referencia exacta (fuerza bruta):', info['optimo_conjunto'], 'coste', info['optimo_coste'])
print('Top-5 fuerza bruta:', info['top5_fuerza_bruta'])
print('Máx. diferencia simulador NumPy vs QAOAAnsatz:', info['max_dif_vs_QAOAAnsatz'])

/home/ubuntu/.local/lib/python3.12/site-packages/scipy/sparse/linalg/_dsolve/linsolve.py:653: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/home/ubuntu/.local/lib/python3.12/site-packages/scipy/sparse/linalg/_matfuncs.py:706: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)
/home/ubuntu/.local/lib/python3.12/site-packages/scipy/sparse/_index.py:174: SparseEfficiencyWarning: Changing the sparsity structure of a csr_matrix is expensive. lil and dok are more efficient.
  self._set_intXint(row, col, x.flat[0])


Referencia exacta (fuerza bruta): ['Ana', 'Enrique'] coste 2.0
Top-5 fuerza bruta: [('011000', 2.0), ('100010', 4.0), ('100100', 4.0), ('110000', 4.0), ('000001', 5.0)]
Máx. diferencia simulador NumPy vs QAOAAnsatz: 8.326672684688674e-16


## Resumen por profundidad (20 reinicios de COBYLA, maxiter=300)

In [2]:
res = reinicios.groupby('p').agg(energia_media=('energy_final','mean'), energia_min=('energy_final','min'), p_opt_media=('p_opt','mean'), p_opt_max=('p_opt','max'), p_visto_shots_media=('p_opt_visto_en_shots','mean'), tiempo_medio_s=('time_s','mean')).round(4)
res

,energia_media,energia_min,p_opt_media,p_opt_max,p_visto_shots_media,tiempo_medio_s
p,,,,,,
1,10.4592,6.0000,0.0260,0.0613,0.8000,0.0496
2,8.9109,6.0000,0.0359,0.1304,0.7000,0.1693
3,8.2806,3.7787,0.0567,0.6877,0.6036,0.2385


## Distribución de medición del mejor reinicio con p=3 (6 000 shots)

In [3]:
dist.head(8)

,bitstring,conjunto,coste_qubo,prob_exacta,conteos_6000_shots,frecuencia_shots
0,011000,"{Ana, Enrique}",2.0,0.687703,4095,0.682500
1,111000,"{Ana, Enrique, Saúl}",7.0,0.229265,1429,0.238167
2,001000,{Ana},7.0,0.024929,144,0.024000
3,011001,"{Mario, Ana, Enrique}",11.0,0.020028,115,0.019167
4,010000,{Enrique},7.0,0.013977,72,0.012000
5,101000,"{Ana, Saúl}",12.0,0.008421,59,0.009833
6,111001,"{Mario, Ana, Enrique, Saúl}",20.0,0.006597,40,0.006667
7,110000,"{Enrique, Saúl}",4.0,0.005071,21,0.003500


## Escalabilidad en grafos Erdős–Rényi (densidad 0.45)

In [4]:
esc = pd.read_csv(m.METRICS / '10_mds_qaoa_escalabilidad.csv')
esc  # generado por `python -m qml_v2.mds_qaoa`

,n,n_aristas,ilp_tamano,greedy_tamano,qubo_optimo,qaoa_mejor_coste_muestreado,qaoa_mejor_p_opt,t_greedy_s,t_ilp_s,t_qaoa_20reinicios_s
0,6,6,2,2,2.0,2.0,0.143858,0.000011,0.023376,4.739300
1,8,11,2,2,4.0,4.0,0.035586,0.000019,0.015745,5.526355
2,10,21,2,2,5.0,5.0,0.049371,0.000014,0.010188,6.325449
3,12,33,2,2,6.0,6.0,0.004494,0.000017,0.007239,8.827871
4,15,50,3,3,8.0,8.0,0.000989,0.000031,0.008227,28.757203


Las figuras `10_qaoa_histograma_medicion.png`, `10_qaoa_convergencia.png` y `10_mds_grafo_solucion.png` se generan con `scripts/generar_figuras_D.py` y `scripts/generar_figuras_categoria_C.py` a partir de estos CSV.